# Session 3 — From One API Call to a Reliable Collection

## Collection certification challenge

Management has expanded the World Bank analysis to:

- 8 countries;
- 15 years (2010–2024);
- one indicator.

A collector was used to produce the dataset supplied with this notebook.

Your task is:

> **Can you certify that the resulting collection is complete?**

AI is allowed. You remain responsible for the evidence supporting your conclusion.


In [5]:
from pathlib import Path
import json

import pandas as pd
import requests

COUNTRIES = ["ESP", "POL", "MAR", "TUR", "FRA", "DEU", "ITA", "PRT"]
YEARS = list(range(2010, 2025))
INDICATOR = "SP.POP.TOTL"
PER_PAGE = 50

BASE_URL = (
    "https://api.worldbank.org/v2/country/"
    + ";".join(COUNTRIES)
    + f"/indicator/{INDICATOR}"
)

DATA_DIR = Path("../data")


## 1. Files supplied

Load the dataset produced by the collector and the raw responses saved during collection.


In [6]:
collector_output = pd.read_csv(
    DATA_DIR / "s3_ai_collector_output.csv"
)

raw_response_paths = [
    DATA_DIR / f"s3_raw_response_{i:02d}.json"
    for i in range(1, 4)
]

raw_responses = [
    json.load(open(path, encoding="utf-8"))
    for path in raw_response_paths
]

print("Rows in collector output:", len(collector_output))
print("Raw response files:", [p.name for p in raw_response_paths])

display(collector_output.head())


Rows in collector output: 120
Raw response files: ['s3_raw_response_01.json', 's3_raw_response_02.json', 's3_raw_response_03.json']


,countryiso3code,country_name,date,indicator_id,indicator_name,value
0,DEU,Germany,2010,SP.POP.TOTL,"Population, total",81842187
1,DEU,Germany,2011,SP.POP.TOTL,"Population, total",81959172
2,DEU,Germany,2012,SP.POP.TOTL,"Population, total",82061715
3,DEU,Germany,2013,SP.POP.TOTL,"Population, total",82154752
4,DEU,Germany,2014,SP.POP.TOTL,"Population, total",82245609


## 2. Collector used

This is the collector that produced the supplied result.



In [7]:
def inherited_collector():
    params = {
        "format": "json",
        "date": f"{min(YEARS)}:{max(YEARS)}",
        "per_page": PER_PAGE,
        "page": 1,
    }

    response = requests.get(BASE_URL, params=params, timeout=10)
    response.raise_for_status()
    first_payload = response.json()

    pages = int(first_payload[0]["pages"])
    total = int(first_payload[0]["total"])

    records = []

    for page in range(1, pages + 1):
        page_to_fetch = min(page, pages - 1)

        params["page"] = page_to_fetch

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=10,
        )
        response.raise_for_status()

        payload = response.json()

        remaining = total - len(records)
        records.extend(payload[1][:remaining])

    return records


## 3. Diagnose

Inspect the collector, the resulting dataset and the supplied raw responses.

Record anything you consider relevant to the certification decision.

Do not modify the collection yet.


In [8]:
# Your diagnostic notes



## 4. Define your evidence

Decide what evidence you would require before signing off on the collection.

For each control, record:

- what you tested;
- the expected result;
- the observed result;
- your interpretation.

Add or remove rows as needed.


In [9]:
evidence = pd.DataFrame(columns=[
    "control",
    "expected",
    "observed",
    "interpretation",
])

evidence


,control,expected,observed,interpretation


## 5. Run your controls

Use the available material to build the evidence required for your decision.


In [10]:
# Your tests here



## 6. Certification decision

Choose one:

- **SIGN OFF**
- **DO NOT SIGN OFF**
- **CANNOT CERTIFY YET**

Explain your decision using the evidence you produced.


**Decision:** ...

**Evidence:** ...

**Remaining uncertainty:** ...


## 7. Repair and re-certification

Only after your diagnosis has been supported by evidence, propose a correction to the collector.

You may use AI.

Then explain what you would test again before signing off on the corrected collection.


In [11]:
# Optional: corrected collector



**Re-certification tests:**  

1. ...
2. ...
3. ...


## 8. AI Audit

- **AI tool used:**
- **Prompt / task given to AI:**
- **What AI proposed:**
- **What you verified yourself:**
- **What you changed, rejected or added:**
- **Remaining limitation:**
